In [1]:
%load_ext autoreload
%autoreload 2
from notebook import *
# if get something about NUMEXPR_MAX_THREADS being set incorrectly, don't worry.  It's not a problem.

# Power consumption and Dark Silicon

## How fast are my CPUs and GPUs?

In [2]:
! lscpu

Architecture:             x86_64
  CPU op-mode(s):         32-bit, 64-bit
  Address sizes:          46 bits physical, 48 bits virtual
  Byte Order:             Little Endian
CPU(s):                   24
  On-line CPU(s) list:    0-23
Vendor ID:                GenuineIntel
  Model name:             13th Gen Intel(R) Core(TM) i7-13700
    CPU family:           6
    Model:                183
    Thread(s) per core:   2
    Core(s) per socket:   16
    Socket(s):            1
    Stepping:             1
    CPU(s) scaling MHz:   57%
    CPU max MHz:          5200.0000
    CPU min MHz:          800.0000
    BogoMIPS:             4224.00
    Flags:                fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge m
                          ca cmov pat pse36 clflush dts acpi mmx fxsr sse sse2 s
                          s ht tm pbe syscall nx pdpe1gb rdtscp lm constant_tsc 
                          art arch_perfmon pebs bts rep_good nopl xtopology nons
                          top_tsc c

In [3]:
! cat /proc/cpuinfo |grep MHz

cpu MHz		: 1805.923
cpu MHz		: 800.000
cpu MHz		: 800.000
cpu MHz		: 800.000
cpu MHz		: 800.000
cpu MHz		: 800.000
cpu MHz		: 800.000
cpu MHz		: 800.000
cpu MHz		: 3027.436
cpu MHz		: 800.000
cpu MHz		: 3040.523
cpu MHz		: 800.000
cpu MHz		: 3075.390
cpu MHz		: 800.000
cpu MHz		: 2983.953
cpu MHz		: 800.000
cpu MHz		: 2197.679
cpu MHz		: 2116.265
cpu MHz		: 800.000
cpu MHz		: 800.000
cpu MHz		: 800.000
cpu MHz		: 800.000
cpu MHz		: 800.000
cpu MHz		: 2197.785


In [11]:
! ssh htseng@eevee "lscpu"

Architecture:                         x86_64
CPU op-mode(s):                       32-bit, 64-bit
Byte Order:                           Little Endian
Address sizes:                        46 bits physical, 48 bits virtual
CPU(s):                               72
On-line CPU(s) list:                  0-71
Thread(s) per core:                   2
Core(s) per socket:                   18
Socket(s):                            2
NUMA node(s):                         2
Vendor ID:                            GenuineIntel
CPU family:                           6
Model:                                85
Model name:                           Intel(R) Xeon(R) Gold 6140 CPU @ 2.30GHz
Stepping:                             4
CPU MHz:                              2340.750
CPU max MHz:                          3700.0000
CPU min MHz:                          1000.0000
BogoMIPS:                             4600.00
Virtualization:                       VT-x
L1d cache:                            1.1 MiB
L1i 

In [12]:
# Operating at lowest freq.
! ssh htseng@eevee "/nfshome/htseng/courses/CSE142/demo/power/lowerMaxFreq.sh; cd /nfshome/htseng/courses/CSE142/demo/power; make -C matrix_mul clean all ;  time sudo likwid-perfctr -g ENERGY matrix_mul/blockmm 1024 16" | grep "STAT"

Sleeping longer as likwid_sleep() called without prior initialization
Sleeping longer as likwid_sleep() called without prior initialization
|   INSTR_RETIRED_ANY STAT   |  FIXC0  | 54673826122 |   0 | 49523239719 | 7.593587e+08 |
| CPU_CLK_UNHALTED_CORE STAT |  FIXC1  | 20600156292 |   0 | 17807280512 | 2.861133e+08 |
|  CPU_CLK_UNHALTED_REF STAT |  FIXC2  | 47380221904 |   0 | 40956720632 | 6.580586e+08 |
|       TEMP_CORE STAT       |   TMP0  |        2138 |  24 |          35 |      29.6944 |
|     PWR_PKG_ENERGY STAT    |   PWR0  |    986.8864 |   0 |    495.8245 |      13.7068 |
|     PWR_PP0_ENERGY STAT    |   PWR1  |           0 |   0 |           0 |            0 |
|    PWR_DRAM_ENERGY STAT    |   PWR3  |    129.6533 |   0 |     85.6994 |       1.8007 |
|  Runtime (RDTSC) [s] STAT |  1289.0016 |  17.9028 |   17.9028 |  17.9028 |
| Runtime unhalted [s] STAT |     8.9564 |        0 |    7.7423 |   0.1244 |
|      Clock [MHz] STAT     | 66003.8142 | 998.7860 | 1001.9252 | 916.7196 |

In [13]:
# Allowing turbo boost
! ssh htseng@eevee "/nfshome/htseng/courses/CSE142/demo/power/restoreMaxFreq.sh; cd /nfshome/htseng/courses/CSE142/demo/power; time sudo likwid-perfctr -g ENERGY matrix_mul/blockmm 1024 16" | grep "STAT"

Sleeping longer as likwid_sleep() called without prior initialization
Sleeping longer as likwid_sleep() called without prior initialization
|   INSTR_RETIRED_ANY STAT   |  FIXC0  | 50459301498 |   0 | 49523236525 | 7.008236e+08 |
| CPU_CLK_UNHALTED_CORE STAT |  FIXC1  | 18260464001 |   0 | 17840082748 | 2.536176e+08 |
|  CPU_CLK_UNHALTED_REF STAT |  FIXC2  | 12465012012 |   0 | 12085460952 | 1.731252e+08 |
|       TEMP_CORE STAT       |   TMP0  |        2300 |  26 |          38 |      31.9444 |
|     PWR_PKG_ENERGY STAT    |   PWR0  |    517.1689 |   0 |    280.5689 |       7.1829 |
|     PWR_PP0_ENERGY STAT    |   PWR1  |           0 |   0 |           0 |            0 |
|    PWR_DRAM_ENERGY STAT    |   PWR3  |     65.8529 |   0 |     40.5609 |       0.9146 |
|  Runtime (RDTSC) [s] STAT |   380.5992 |   5.2861 |    5.2861 |   5.2861 |
| Runtime unhalted [s] STAT |     7.9398 |        0 |    7.7568 |   0.1103 |
|      Clock [MHz] STAT     | 33412.1270 | 997.3143 | 3395.0566 | 464.0573 |

In [8]:
# Operating at lowest freq.
! ssh htseng@eevee "/nfshome/htseng/courses/CSE142/demo/power/lowerMaxFreq.sh; cd /nfshome/htseng/courses/CSE142/demo/power/popcounts; make clean; make; time sudo likwid-perfctr -g ENERGY ./popcount_D" | grep "STAT"

Sleeping longer as likwid_sleep() called without prior initialization
Sleeping longer as likwid_sleep() called without prior initialization
|   INSTR_RETIRED_ANY STAT   |  FIXC0  | 80118744646 |   0 | 80002272949 | 2.503711e+09 |
| CPU_CLK_UNHALTED_CORE STAT |  FIXC1  | 21473661358 |   0 | 21389051661 | 6.710519e+08 |
|  CPU_CLK_UNHALTED_REF STAT |  FIXC2  | 48315649176 |   0 | 48125278584 | 1.509864e+09 |
|       TEMP_CORE STAT       |   TMP0  |         730 |  18 |          27 |      22.8125 |
|     PWR_PKG_ENERGY STAT    |   PWR0  |   1032.6894 |   0 |    534.1400 |      32.2715 |
|     PWR_PP0_ENERGY STAT    |   PWR1  |           0 |   0 |           0 |            0 |
|    PWR_DRAM_ENERGY STAT    |   PWR3  |     79.5384 |   0 |     72.1068 |       2.4856 |
|  Runtime (RDTSC) [s] STAT |   857.4368 |  26.7949 |  26.7949 |  26.7949 |
| Runtime unhalted [s] STAT |    11.9299 |        0 |  11.8829 |   0.3728 |
|      Clock [MHz] STAT     | 16000.1414 | 798.4320 | 801.3755 | 500.0044 |
| 

In [9]:
# Allowing turbo boost
! ssh htseng@eevee "/nfshome/htseng/courses/CSE142/demo/power/restoreMaxFreq.sh; cd /nfshome/htseng/courses/CSE142/demo/power/popcounts;  sudo likwid-perfctr -g ENERGY ./popcount_D" | grep "STAT"

Sleeping longer as likwid_sleep() called without prior initialization
Sleeping longer as likwid_sleep() called without prior initialization
|   INSTR_RETIRED_ANY STAT   |  FIXC0  | 80005178427 |   0 | 80002268036 | 2.500162e+09 |
| CPU_CLK_UNHALTED_CORE STAT |  FIXC1  | 21387905485 |   0 | 21383241417 | 6.683720e+08 |
|  CPU_CLK_UNHALTED_REF STAT |  FIXC2  | 12880339128 |   0 | 12869923608 | 4.025106e+08 |
|       TEMP_CORE STAT       |   TMP0  |         765 |  19 |          28 |      23.9062 |
|     PWR_PKG_ENERGY STAT    |   PWR0  |    382.4405 |   0 |    207.3791 |      11.9513 |
|     PWR_PP0_ENERGY STAT    |   PWR1  |           0 |   0 |           0 |            0 |
|    PWR_DRAM_ENERGY STAT    |   PWR3  |     46.4689 |   0 |     27.6189 |       1.4522 |
|  Runtime (RDTSC) [s] STAT |   229.1744 |   7.1617 |    7.1617 |   7.1617 |
| Runtime unhalted [s] STAT |    11.8821 |        0 |   11.8796 |   0.3713 |
|      Clock [MHz] STAT     | 13767.5719 | 799.6714 | 2990.6788 | 430.2366 |

In [38]:
# Operating at lowest freq.
! ssh htseng@eevee "/nfshome/htseng/courses/CSE142/demo/power/lowerMaxFreq.sh; cd /nfshome/htseng/courses/CSE142/demo/power;  time sudo likwid-perfctr -g ENERGY matrix_mul/blockmm_pthread 1024 16 32" | grep "STAT"

Sleeping longer as likwid_sleep() called without prior initialization
Sleeping longer as likwid_sleep() called without prior initialization
|   INSTR_RETIRED_ANY STAT   |  FIXC0  |  9319801705 |   0 |  9316611709 | 2.912438e+08 |
| CPU_CLK_UNHALTED_CORE STAT |  FIXC1  |  6364606801 |   0 |  6359227008 | 1.988940e+08 |
|  CPU_CLK_UNHALTED_REF STAT |  FIXC2  | 14320359432 |   0 | 14308253928 | 4.475112e+08 |
|       TEMP_CORE STAT       |   TMP0  |         672 |  16 |          25 |           21 |
|     PWR_PKG_ENERGY STAT    |   PWR0  |    309.0314 |   0 |    160.0001 |       9.6572 |
|     PWR_PP0_ENERGY STAT    |   PWR1  |           0 |   0 |           0 |            0 |
|    PWR_DRAM_ENERGY STAT    |   PWR3  |     23.2114 |   0 |     20.7645 |       0.7254 |
|  Runtime (RDTSC) [s] STAT |   256.5728 |   8.0179 |   8.0179 |   8.0179 |
| Runtime unhalted [s] STAT |     3.5361 |        0 |   3.5329 |   0.1105 |
|      Clock [MHz] STAT     | 13598.3625 | 798.9159 | 800.5734 | 424.9488 |
| 

In [39]:
# Allowing turbo boost
! ssh htseng@eevee "/nfshome/htseng/courses/CSE142/demo/power/restoreMaxFreq.sh; cd /nfshome/htseng/courses/CSE142/demo/power;  time sudo likwid-perfctr -g ENERGY matrix_mul/blockmm_pthread 1024 16 4" | grep "STAT"

Sleeping longer as likwid_sleep() called without prior initialization
Sleeping longer as likwid_sleep() called without prior initialization
|   INSTR_RETIRED_ANY STAT   |  FIXC0  | 9321315256 |   0 | 9316610267 | 2.912911e+08 |
| CPU_CLK_UNHALTED_CORE STAT |  FIXC1  | 6406740239 |   0 | 6400177256 | 2.002106e+08 |
|  CPU_CLK_UNHALTED_REF STAT |  FIXC2  | 3872239056 |   0 | 3857685408 | 1.210075e+08 |
|       TEMP_CORE STAT       |   TMP0  |        703 |  17 |         26 |      21.9688 |
|     PWR_PKG_ENERGY STAT    |   PWR0  |   117.4410 |   0 |    64.5992 |       3.6700 |
|     PWR_PP0_ENERGY STAT    |   PWR1  |          0 |   0 |          0 |            0 |
|    PWR_DRAM_ENERGY STAT    |   PWR3  |    13.8510 |   0 |     8.2689 |       0.4328 |
|  Runtime (RDTSC) [s] STAT |    69.7344 |   2.1792 |    2.1792 |   2.1792 |
| Runtime unhalted [s] STAT |     3.5594 |        0 |    3.5557 |   0.1112 |
|      Clock [MHz] STAT     | 20211.6724 | 799.6763 | 2986.3295 | 631.6148 |
|          CP